In [2]:
import os
import sys
import math
from statistics import mean
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from control import * 
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from local_occupancy import sensor_measurements, sensor_distances
from local_occupancy import thymio_coords, sensor_pos_from_center, sensor_angles

from local import * 

In [4]:
from tdmclient import ClientAsync

client = ClientAsync()
node = await client.wait_for_node()
await node.lock()


Node 848cf03d-8bca-4685-9cca-f00b771ebd9e

In [6]:
grid = LocalOccupancyGrid(
    size_m=0.6,
    resolution_m=0.01,
    hit_inc=0.6,
    decay=0.98,
    robot_radius_m=0.09  # tweak between 0.08–0.10

)

cfg = LocalNavConfig(
    lookahead_dist_m=0.15,
    max_lookahead_points=30,
    occ_threshold=0.3,
    rep_influence_radius=0.30,
    k_att=1.0,
    k_rep=1.5,
    virt_goal_dist_m=0.10
)

navigator = LocalNavigator(grid, cfg)

gains = GoToGoalGains(
    Kv=2.0,
    Komega=3.0,
    v_max=0.18,
    w_max=2.0
)
kin   = ThymioKinematics()
g2g   = GoToGoalController(gains, kin)


In [7]:
path_length = 0.8
num_points  = 60
x_path = np.linspace(0.0, path_length, num_points)
y_path = np.zeros(num_points)
path   = np.vstack((x_path, y_path)).T


In [9]:
def motors(l_speed=0, r_speed=0):
    return {
        "motor.left.target":  [int(l_speed)],
        "motor.right.target": [int(r_speed)],
    }

async def follow_segment_with_avoid_debug(path,
                                          dt=0.10,
                                          max_time_s=20.0,
                                          goal_tol_m=0.02):
    await node.wait_for_variables({"prox.horizontal"})

    # Start pose at path start, facing along +x
    pose = np.array([path[0, 0], path[0, 1], 0.0], dtype=float)
    final_goal = path[-1]
    n_steps = int(max_time_s / dt)

    print("Starting path following with obstacle avoidance (DEBUG)...")

    for k in range(n_steps):
        # 1) Read Thymio sensors
        sensor_vals = list(node.v.prox.horizontal)

        # 2) Update occupancy grid
        grid.update_from_sensor_vals(sensor_vals)

        # DEBUG: grid stats
        max_occ = np.max(grid.grid)
        num_thr = np.sum(grid.grid > cfg.occ_threshold)

        # 3) Find look-ahead (LA) point on path
        LA_world, _ = navigator._find_lookahead_point(pose, path)
        if LA_world is None:
            print("No LA point, stopping.")
            break

        # 4) Express LA in robot frame
        p_LA_robot = navigator.world_to_robot(pose, LA_world)

        # 5) Attractive vector towards LA
        F_att = p_LA_robot.astype(float)
        norm_att = np.linalg.norm(F_att)
        if norm_att > 1e-6:
            F_att = (cfg.k_att / norm_att) * F_att

        # 6) Repulsive vector from grid
        F_rep = navigator._repulsive_vector()
        norm_rep = np.linalg.norm(F_rep)

        # 7) Total vector in robot frame
        F_tot = F_att + F_rep
        norm_tot = np.linalg.norm(F_tot)
        if norm_tot < 1e-6:
            F_tot = np.array([0.0, 1.0], dtype=float)
            norm_tot = 1.0

        dir_robot = F_tot / norm_tot
        virt_robot = cfg.virt_goal_dist_m * dir_robot

        # 8) Convert virtual goal back to world frame
        virt_world = navigator.robot_to_world(pose, virt_robot)

        # 9) Controller: go to virtual goal
        uL, uR, info = g2g.compute_motor_commands(pose, virt_world,
                                                  stop_if_close=False,
                                                  tol=goal_tol_m)

        await node.set_variables(motors(uL, uR))

        # 10) Integrate pose from commanded speeds (open-loop)
        r = kin.wheel_radius
        L = kin.half_axle
        phi_L = uL / kin.motor_gain
        phi_R = uR / kin.motor_gain
        v = r * (phi_R + phi_L) / 2.0
        w = r * (phi_R - phi_L) / (2.0 * L)

        pose[0] += v * math.cos(pose[2]) * dt
        pose[1] += v * math.sin(pose[2]) * dt
        pose[2]  = wrap_to_pi(pose[2] + w * dt)

        # 11) Debug prints every few steps
        if k % 5 == 0:
            delta_VG_LA = np.linalg.norm(virt_world - LA_world)
            print(f"step {k} | dist_to_goal={math.hypot(final_goal[0]-pose[0], final_goal[1]-pose[1]):.3f} m")
            print(f"  sensors={sensor_vals}")
            print(f"  max_occ={max_occ:.3f}, cells>thr={num_thr}")
            print(f"  LA_world=({LA_world[0]:.3f},{LA_world[1]:.3f})")
            print(f"  VG_world=({virt_world[0]:.3f},{virt_world[1]:.3f}), |VG-LA|={delta_VG_LA:.3f}")
            print(f"  F_att=({F_att[0]:.3f},{F_att[1]:.3f}), F_rep=({F_rep[0]:.3f},{F_rep[1]:.3f})")
            print(f"  uL={uL}, uR={uR}")
            print("")

        # 12) Check if we reached final goal
        dx = final_goal[0] - pose[0]
        dy = final_goal[1] - pose[1]
        if math.hypot(dx, dy) < goal_tol_m:
            print(f"Reached goal within {goal_tol_m*100:.1f} cm at step {k}")
            break

        await client.sleep(dt)

    await node.set_variables(motors(0, 0))
    print("Stopped.")


In [10]:
await follow_segment_with_avoid_debug(path)


Starting path following with obstacle avoidance (DEBUG)...
step 0 | dist_to_goal=0.782 m
  sensors=[0, 1497, 1796, 0, 0, 0, 2470]
  max_occ=1.000, cells>thr=613
  LA_world=(0.163,0.000)
  VG_world=(-0.099,0.014), |VG-LA|=0.262
  F_att=(0.000,1.000), F_rep=(159.605,-1117.644)
  uL=38, uR=125

step 5 | dist_to_goal=0.713 m
  sensors=[0, 0, 0, 0, 0, 0, 0]
  max_occ=0.941, cells>thr=651
  LA_world=(0.244,0.000)
  VG_world=(0.014,-0.042), |VG-LA|=0.234
  F_att=(-0.935,0.354), F_rep=(159.605,-1117.644)
  uL=38, uR=125

step 10 | dist_to_goal=0.727 m
  sensors=[3253, 1368, 0, 0, 0, 0, 0]
  max_occ=1.000, cells>thr=827
  LA_world=(0.258,0.000)
  VG_world=(0.089,0.019), |VG-LA|=0.170
  F_att=(-0.482,-0.876), F_rep=(991.559,-1885.872)
  uL=38, uR=125

step 15 | dist_to_goal=0.809 m
  sensors=[0, 0, 0, 0, 0, 0, 0]
  max_occ=0.980, cells>thr=866
  LA_world=(0.258,0.000)
  VG_world=(0.106,0.113), |VG-LA|=0.189
  F_att=(0.509,-0.861), F_rep=(1462.259,-2337.348)
  uL=38, uR=125

step 20 | dist_to_goa

CancelledError: 

In [11]:
await node.set_variables(motors(0, 0))
